# Sudo Security

## The `root` account 

On macOS, you can use different accounts to log in. Those accounts can be `standard accounts` or `admin accounts`. There are some things that you can do with an admin account that you can't do with a standard account. For example, only someone with an admin account can create an account for another user.

On some operating systems derived from Unix, it is possible to log in as `root` user who is even higher up the permissions hierarchy. In most operating systems, the root can do anything. On macOS, no one can log in as `root`. Root is an identity that you can take on only briefly. When you do, on macOS you are not all powerful. 

In the terminal, instead of running a command as your user, you can run it as the `root` user by prefacing a command with `sudo`, which stands for &ldquo;substitute user do.&rdquo; 

For example, you can run the shell command 
    
    cat filename 

as yourself, using your own permissions. Or you can run this command as the `root` by entering  

    sudo cat filename

## The `sudo` command 

The default in macOS is that all admin users are able to run `sudo`. Almost everyone logs in as an admin user, so almost everyone can use `sudo`. 

From these two facts, you might think that having the `sudo` command makes no difference to security. But in fact, the `sudo` command is an essential and very powerful part of the macOS security model. It limits what you can do, to limit even more tightly what malware can do. The fundamental security weakness of Windows is that it has no such command. 

The `sudo` command makes a difference because the user has authenticate before the command it prefaces will run. In the base configuration, a command prefaced by `sudo` won't run unless users types in their login password into a form that the OS presents. 

## The threat 

Why force someone to log in a second time? Because someone who is logged in may inadvertently let someone else act take on the privileges of the logged in user. 

To keep the roles straight, let's use some names. Suppose that Bob is a software developer with an admin account on his Mac. Mallory has written some malware, a Python script called `harden_ssh.py`. Mallory includes in a library he calls `Security_Tools` that he publishes on `https://pypi.org`. 

If Mallory's script runs on Bob's computer, it looks for a `.ssh` directory in Bob's user's home directory. If it finds one, it tries to read any secret keys that are stored there. It also reads the `config` file to to learn which website where each key works with. The script uses one of many possible methods to send what it learns to a website that Mallory controls. It also tries to use git and ssh to submit Mallory's malicious `harden_ssh.py` file to any repositories where Bob is a contributor. 

You should be able to convince yourself that it would be very easy to write such a script. It would take a little bit of social engineering to get Bob to install Mallory's `Security_Tools` library and run the `harden_ssh.py` script, but if Mallory targets 1000 people, it is not hard to believe that at least one of them, plausible that one of them, the person we are calling Bob, runs the script. 

If Bob runs the script, it runs with Bob's privileges. The permissions on her ssh secret keys may say that only Bob can read these files. This clearly is not a problem for Mallory's script because it runs with Bob's privileges. 

## How `sudo` helps

Suppose that Bob has configured SSH so that it makes a connection only if the command is prefaced with `sudo`. Mallory's script might run silently, but if it runs a `git push` command, it won't run unless Bob approves it. If Bob is paying attention, he will not do as he is told. He will recognize that he did not do anything to trigger any request that requires that he re-authenticate. He won't help the malware propagate over SSH. He should disconnect from the internet and start looking for the malware that made the request. 

The `sudo` command helps because it requires an action that can't be implemented by software. A person at the keyboard has to enter the keystrokes for Bob's login password. Apple has been careful to ensure that Mallory's Python script can't simulate the entry of the password. As we will see, Apple has also taken steps that make it possible for Bob to make sure that Mallory's script can't use a keylogger to record all of Bob's keystrokes and discover his login password. 

## Outline of details

### With macOS defaults
- You can't run sudo from JupyterLab;
- You can run a sudo command from the JupyterLab Terminal BUT YOU SHOULD NOT; 
- You can run a sudo command from the macOS Terminal BUT YOU SHOULD NOT UNTIL YOU TURN ON Secure Keyboard Entry in Terminal;
- Once Secure Keyboard Entry is on, you can use a sudo command to switch from the default of password authorization for `sudo` commands to a biometric touch as authorization.

Shifting to a biometric touch is a rare instance of better security that is also more convenient. Of course, you do have to set up TouchID with macOS by registering at least one fingerprint. And you have to be using macOS hardware with a touch sensor. 

### Once TouchID is set as a way to authorize any `sudo` commands:
- You can safely run sudo commands from a notebook in JupyterLab or from the JupyterLab terminal;
- You will not need to keep Secure Keyboard Entry on in Terminal;
- In fact, it will be more convenient for you if you turn off Secure Keyboard Entry. 

## Text Files that Change Settings 

Outline

##### 1. Put a file called `sudo-local` into `/etc/pam.d`

##### 2. Put some files into `/etc/sudoers.d`

##### 3. Put a command you can run into `/usr/local/bin`
